<a href="https://colab.research.google.com/github/TAUforPython/machinelearning/blob/main/example_MAMBA_LLM_State_Space_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

https://github.com/state-spaces/mamba

In [1]:
# @title Setup and Imports
# Cell 1: Setup Environment and Install Dependencies
import os
import sys
import math
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

# Install necessary packages for Colab
# We'll use the official mamba-ssm package
!pip install mamba-ssm --quiet
!pip install causal-conv1d>=1.4.0 --quiet # Required for mamba-ssm

# Ensure the environment is compatible (Colab GPUs are NVIDIA)
gpu_available = torch.cuda.is_available()
print(f"CUDA Available: {gpu_available}")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.7/121.7 kB 2.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─

NameError: name 'torch' is not defined

In [2]:
!pip install torch -q

In [4]:
!pip install mamba-ssm --quiet

  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.


In [5]:

import torch
import torch.nn as nn
import torch.optim as optim
from mamba_ssm import Mamba, Mamba2 # Import the official Mamba modules

# Set device
device = torch.device("cuda" if gpu_available else "cpu")
print(f"Using device: {device}")

ModuleNotFoundError: No module named 'mamba_ssm'

In [ ]:
# @title Define the True LTI System
# Cell 2: Define the Continuous-Time LTI System Dynamics

def lti_system_derivatives(t, state, A_matrix, B_matrix, input_signal_func):
  """
  Defines the derivatives for the continuous-time LTI system:
  dx/dt = Ax + Bu
  """
  x = state # State vector [x1, x2]
  u = input_signal_func(t) # Input signal u(t)

  dx_dt = A_matrix @ x + B_matrix * u
  return dx_dt

def simulate_lti_system(A, B, u_func, x0, t_eval, method='RK45'):
  """
  Simulates the LTI system dx/dt = Ax + Bu using scipy.integrate.solve_ivp.
  """
  from scipy.integrate import solve_ivp

  def deriv(t, y):
      return lti_system_derivatives(t, y, A, B, u_func)

  sol = solve_ivp(deriv, [t_eval[0], t_eval[-1]], x0, t_eval=t_eval, method=method, rtol=1e-8)
  return sol.y.T, sol.t # Return state trajectories and time points

# --- System Definition ---
# Define a stable second-order LTI system matrix A (eigenvalues have negative real parts)
# Example: A system representing a damped oscillator
lambda_real = -0.5  # Real part of eigenvalue (negative for stability)
lambda_imag = 1.0   # Imaginary part of eigenvalue (frequency)

# Create A matrix with the desired eigenvalues
# Complex eigenvalues lambda = -0.5 +/- 1.0j lead to oscillatory decay
A_true = np.array([
    [lambda_real, -lambda_imag],
    [lambda_imag,  lambda_real]
])
print(f"True System Matrix A:\n{A_true}")

# Calculate eigenvalues to confirm stability
eigenvals_A = np.linalg.eigvals(A_true)
print(f"Eigenvalues of A: {eigenvals_A}")
print(f"System is stable: {np.all(eigenvals_A.real < 0)}")

# Define input matrix B (how input affects the system)
B_true = np.array([1.0, 0.5]) # Shape (2,)
print(f"True Input Matrix B: {B_true}")

# Define an input signal u(t) (e.g., a decaying sine wave)
def u_signal(t):
  # This creates a rich signal for excitation
  return 2.0 * np.exp(-0.1 * t) * np.sin(0.5 * t)

# Simulation parameters
dt = 0.01
T_max = 10.0
t_vals = np.arange(0.0, T_max + dt, dt)
print(f"Simulation time span: [{t_vals[0]}, {t_vals[-1]}] seconds, with dt = {dt}")

# Initial conditions for the system
x0_conditions = [
    [1.0, 0.0],  # Initial state 1
    [-1.0, 1.0], # Initial state 2
    [0.0, -1.0], # Initial state 3
]
num_trajectories = len(x0_conditions)

# Simulate the system for all initial conditions
true_states_list = []
for i, x0 in enumerate(x0_conditions):
    x_true, _ = simulate_lti_system(A_true, B_true, u_signal, np.array(x0), t_vals)
    true_states_list.append(x_true)

# Combine into a single array for easier handling (time_steps, num_trajectories, state_dim)
all_true_states = np.stack(true_states_list, axis=1) # Shape: (len(t_vals), num_trajectories, 2)
print(f"Simulated true states shape: {all_true_states.shape}")

In [ ]:
# @title Visualize True System Behavior
# Cell 3: Visualize Phase Trajectories and Step Response of the True System

# Plot Phase Trajectories
plt.figure(figsize=(10, 8))
for i in range(num_trajectories):
    x1_traj = all_true_states[:, i, 0]
    x2_traj = all_true_states[:, i, 1]
    plt.plot(x1_traj, x2_traj, label=f'Trajectory {i+1}', linewidth=2, marker='o', markersize=2)
    plt.scatter(x0_conditions[i][0], x0_conditions[i][1], color='red', s=100, zorder=5, label=f'Init {i+1}' if i == 0 else "") # Mark initial points

plt.title('Phase Portrait of the True LTI System')
plt.xlabel('State $x_1$')
plt.ylabel('State $x_2$')
plt.grid(True)
plt.legend()
plt.axis('equal') # Equal aspect ratio for phase plot
plt.show()

# Plot Step Response (example for one initial condition)
plt.figure(figsize=(12, 4))
x0_example_idx = 0
x0_example = x0_conditions[x0_example_idx]
x_true_example = all_true_states[:, x0_example_idx, :]

plt.subplot(1, 3, 1)
plt.plot(t_vals, x_true_example[:, 0], label='$x_1(t)$', linewidth=2)
plt.plot(t_vals, x_true_example[:, 1], label='$x_2(t)$', linewidth=2)
plt.title(f'Step Response (Init: {x0_example})')
plt.xlabel('Time (s)')
plt.ylabel('State Value')
plt.grid(True)
plt.legend()

plt.subplot(1, 3, 2)
plt.plot(t_vals, u_signal(t_vals), label='$u(t)$', color='orange', linewidth=2)
plt.title('Input Signal $u(t)$')
plt.xlabel('Time (s)')
plt.ylabel('Signal Value')
plt.grid(True)
plt.legend()

# Plot Input vs States
plt.subplot(1, 3, 3)
plt.plot(x_true_example[:, 0], x_true_example[:, 1], label='Trajectory', linewidth=2)
plt.scatter(x0_example[0], x0_example[1], color='red', s=100, zorder=5, label='Initial Point')
plt.title('State Space Projection')
plt.xlabel('$x_1$')
plt.ylabel('$x_2$')
plt.grid(True)
plt.legend()
plt.axis('equal')

plt.tight_layout()
plt.show()

In [ ]:
# @title Define the Mamba-Based Estimator Model
# Cell 4: Define the Mamba-Inspired Model for State Estimation

class StateEstimatorMamba(nn.Module):
    """
    A model inspired by the Mamba architecture to estimate the hidden state of an LTI system.
    Inputs: Observed signals (here, just the input u(t) and potentially a noisy observation of x1 or x2).
           For simplicity, we use only u(t) as input, assuming direct state observation is not available.
    Outputs: Estimated state trajectory x_hat(t) = [x_hat1(t), x_hat2(t)]
    """
    def __init__(self, input_dim, hidden_dim, state_dim, num_blocks=2):
        super().__init__()
        self.input_dim = input_dim
        self.state_dim = state_dim
        self.hidden_dim = hidden_dim # Internal model dimension

        # Optional: Embed the scalar input u(t) into a higher-dimensional space
        self.input_proj = nn.Linear(input_dim, hidden_dim)

        # Stack Mamba blocks (using the official implementation)
        # Note: Mamba blocks are typically stateless regarding the sequence processing context
        # within a single forward pass. The recurrent nature is for *generation*.
        # For *estimation* from a full sequence, we process the sequence, potentially
        # using the final output or all outputs depending on the task.
        # Here, we adapt the block for a 'seq-to-seq' mapping.
        self.mamba_blocks = nn.ModuleList([
            Mamba(d_model=hidden_dim, d_state=16, d_conv=4, expand=2) # Typical Mamba config
            for _ in range(num_blocks)
        ])

        # Final projection layer to map back to the state dimension
        self.output_proj = nn.Linear(hidden_dim, state_dim)

        # Optional normalization layer (as mentioned in Mamba paper Sec 3.4)
        self.norm = nn.LayerNorm(hidden_dim)

    def forward(self, u_signal_batch):
        """
        Args:
          u_signal_batch: Tensor of shape (batch_size, seq_len, input_dim)
                          e.g., (num_trajectories, len(t_vals), 1) for scalar u(t)
        Returns:
          x_hat_batch: Tensor of shape (batch_size, seq_len, state_dim)
                       Estimated state trajectory for each batch item.
        """
        # 1. Project input
        x = self.input_proj(u_signal_batch) # (B, L, H)

        # 2. Process through Mamba blocks
        for block in self.mamba_blocks:
            x_res = x # Residual connection
            x = block(x) # x shape remains (B, L, H)
            x = x + x_res # Add residual
            x = self.norm(x) # Apply normalization

        # 3. Project to output state dimension
        x_hat_batch = self.output_proj(x) # (B, L, state_dim)

        return x_hat_batch

# --- Model Configuration ---
input_dim = 1  # Dimension of the input signal u(t)
state_dim = 2  # Dimension of the state vector x(t) = [x1, x2]
hidden_dim = 64 # Internal dimension for the Mamba model
num_blocks = 2 # Number of Mamba blocks to stack

# Instantiate the estimator model
estimator_model = StateEstimatorMamba(
    input_dim=input_dim,
    hidden_dim=hidden_dim,
    state_dim=state_dim,
    num_blocks=num_blocks
).to(device)

print(f"Estimator Model:\n{estimator_model}")
print(f"Total parameters: {sum(p.numel() for p in estimator_model.parameters())}")

In [ ]:
# @title Prepare Data and Train the Estimator
# Cell 5: Prepare Training Data and Train the Mamba Estimator

# Prepare the input data (u signal) and target data (true states) for training
# Shape: (batch_size, seq_len, features)
u_signal_tensor = torch.tensor(u_signal(t_vals), dtype=torch.float32).unsqueeze(-1).unsqueeze(0) # (1, L, 1)
# Repeat for each trajectory to create batch
u_signal_batch = u_signal_tensor.repeat(num_trajectories, 1, 1).to(device) # (num_trajectories, L, 1)

true_states_tensor = torch.tensor(all_true_states, dtype=torch.float32).to(device) # (L, num_trajectories, 2)
true_states_batch = true_states_tensor.transpose(0, 1).to(device) # (num_trajectories, L, 2)

print(f"Input batch shape: {u_signal_batch.shape}")
print(f"Target batch shape: {true_states_batch.shape}")

# Define Loss and Optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(estimator_model.parameters(), lr=0.001, weight_decay=1e-5)

# Training Loop
num_epochs = 500
losses = []

estimator_model.train()
for epoch in range(num_epochs):
    optimizer.zero_grad()

    # Forward pass
    x_hat_pred = estimator_model(u_signal_batch) # (num_trajectories, L, 2)

    # Calculate loss (MSE between predicted and true states)
    loss = criterion(x_hat_pred, true_states_batch)

    # Backward pass and optimization
    loss.backward()
    optimizer.step()

    losses.append(loss.item())

    if (epoch + 1) % 100 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.6f}')

# Plot training loss
plt.figure(figsize=(10, 4))
plt.plot(losses)
plt.title('Training Loss (MSE)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True)
plt.show()

In [ ]:
# @title Evaluate the Estimator and Visualize Results
# Cell 6: Evaluate the Model and Compare True vs Estimated Trajectories

estimator_model.eval()
with torch.no_grad():
    x_hat_estimated_batch = estimator_model(u_signal_batch) # (num_trajectories, L, 2)

# Convert back to numpy for plotting
x_hat_estimated_np = x_hat_estimated_batch.cpu().numpy() # (num_trajectories, L, 2)
true_states_np = true_states_batch.cpu().numpy() # (num_trajectories, L, 2)

# --- Visualization ---
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Plot 1: Phase Portrait Comparison (Trajectory 1)
traj_idx = 0
axes[0, 0].plot(true_states_np[traj_idx, :, 0], true_states_np[traj_idx, :, 1], label='True $x(t)$', linewidth=2, color='blue')
axes[0, 0].plot(x_hat_estimated_np[traj_idx, :, 0], x_hat_estimated_np[traj_idx, :, 1], label='Estimated $\hat{x}(t)$', linewidth=2, linestyle='--', color='red')
axes[0, 0].scatter(true_states_np[traj_idx, 0, 0], true_states_np[traj_idx, 0, 1], color='blue', s=100, zorder=5, label='Init True')
axes[0, 0].scatter(x_hat_estimated_np[traj_idx, 0, 0], x_hat_estimated_np[traj_idx, 0, 1], color='red', s=100, zorder=5, label='Init Est.')
axes[0, 0].set_title(f'Phase Plot (Trajectory {traj_idx+1}): True vs Estimated')
axes[0, 0].set_xlabel('$x_1$')
axes[0, 0].set_ylabel('$x_2$')
axes[0, 0].legend()
axes[0, 0].grid(True)
axes[0, 0].axis('equal')

# Plot 2: State Evolution Comparison (Trajectory 1)
axes[0, 1].plot(t_vals, true_states_np[traj_idx, :, 0], label='True $x_1(t)$', linewidth=2, color='blue')
axes[0, 1].plot(t_vals, x_hat_estimated_np[traj_idx, :, 0], label='Estimated $\hat{x}_1(t)$', linewidth=2, linestyle='--', color='red')
axes[0, 1].plot(t_vals, true_states_np[traj_idx, :, 1], label='True $x_2(t)$', linewidth=2, color='green')
axes[0, 1].plot(t_vals, x_hat_estimated_np[traj_idx, :, 1], label='Estimated $\hat{x}_2(t)$', linewidth=2, linestyle='--', color='orange')
axes[0, 1].set_title(f'State Evolution (Trajectory {traj_idx+1}): True vs Estimated')
axes[0, 1].set_xlabel('Time (s)')
axes[0, 1].set_ylabel('State Value')
axes[0, 1].legend()
axes[0, 1].grid(True)

# Plot 3: Phase Portrait Comparison (Trajectory 2)
traj_idx = 1
axes[1, 0].plot(true_states_np[traj_idx, :, 0], true_states_np[traj_idx, :, 1], label='True $x(t)$', linewidth=2, color='blue')
axes[1, 0].plot(x_hat_estimated_np[traj_idx, :, 0], x_hat_estimated_np[traj_idx, :, 1], label='Estimated $\hat{x}(t)$', linewidth=2, linestyle='--', color='red')
axes[1, 0].scatter(true_states_np[traj_idx, 0, 0], true_states_np[traj_idx, 0, 1], color='blue', s=100, zorder=5, label='Init True')
axes[1, 0].scatter(x_hat_estimated_np[traj_idx, 0, 0], x_hat_estimated_np[traj_idx, 0, 1], color='red', s=100, zorder=5, label='Init Est.')
axes[1, 0].set_title(f'Phase Plot (Trajectory {traj_idx+1}): True vs Estimated')
axes[1, 0].set_xlabel('$x_1$')
axes[1, 0].set_ylabel('$x_2$')
axes[1, 0].legend()
axes[1, 0].grid(True)
axes[1, 0].axis('equal')

# Plot 4: Overall MSE Error over time
mse_per_timestep = np.mean((true_states_np - x_hat_estimated_np)**2, axis=(0, 2)) # Average over trajectories and state dims
axes[1, 1].plot(t_vals, mse_per_timestep, label='MSE $(x - \hat{x})^2$', linewidth=2, color='purple')
axes[1, 1].set_title('Mean Squared Error (MSE) over Time Steps')
axes[1, 1].set_xlabel('Time (s)')
axes[1, 1].set_ylabel('MSE')
axes[1, 1].grid(True)
axes[1, 1].legend()

plt.tight_layout()
plt.show()

# --- Accuracy Check ---
final_mse_per_trajectory = np.mean((true_states_np - x_hat_estimated_np)**2, axis=1) # (num_trajectories, state_dim)
final_mse_total = np.mean(final_mse_per_trajectory) # Scalar overall MSE

print("\n--- Accuracy Check ---")
print(f"Final Overall MSE (averaged over time, trajectories, and states): {final_mse_total:.6f}")
print(f"Final MSE per trajectory (averaged over time and states):\n{np.mean(final_mse_per_trajectory, axis=1)}")
print(f"Final MSE per state (averaged over time and trajectories):\n{np.mean(final_mse_per_trajectory, axis=0)}")

# Example: Print stats for the first trajectory
traj_idx = 0
print(f"\nDetailed stats for Trajectory {traj_idx+1}:")
print(f"  - MSE for x1: {final_mse_per_trajectory[traj_idx, 0]:.6f}")
print(f"  - MSE for x2: {final_mse_per_trajectory[traj_idx, 1]:.6f}")
print(f"  - Mean MSE for traj {traj_idx+1}: {np.mean(final_mse_per_trajectory[traj_idx]):.6f}")